In [1]:
from datetime import datetime
from typing import Any, Dict, List, Optional

import pandas as pd

In [4]:
def load_data(path: str) -> pd.DataFrame:
    """Load a CSV into a pandas DataFrame and log its shape."""
    # log(f"Loading dataset: {path}")
    df = pd.read_csv(path)
    # log(f"Loaded {df.shape[0]} rows × {df.shape[1]} cols")
    return df

In [5]:
# Step 1 — ID Column Detection
def detect_id_columns(df):
    id_columns = []

    for col in df.columns:
        unique_ratio = df[col].nunique() / len(df)

        if unique_ratio > 0.95:
            id_columns.append(col)

        if "id" in col.lower():
            id_columns.append(col)

    return list(set(id_columns))

# Step 2 — Target Candidate Scoring
def score_target_candidates(df, id_columns):

    scores = {}

    for col in df.columns:

        if col in id_columns:
            continue

        score = 0
        unique_ratio = df[col].nunique() / len(df)

        # ---- Signal 1: Low Cardinality ----
        if unique_ratio < 0.1:
            score += 3
        elif unique_ratio < 0.3:
            score += 1

        # ---- Signal 2: Column Name ----
        target_keywords = ["target", "label", "class", "output", "result", "status", "y", "outcome"]

        if any(keyword in col.lower() for keyword in target_keywords):
            score += 4

        # ---- Signal 3: Last Column Bias ----
        if col == df.columns[-1]:
            score += 2

        # ---- Signal 4: Datatype Pattern ----
        if not pd.api.types.is_datetime64_any_dtype(df[col]):
            score += 2

        scores[col] = score

    return scores

# Step 3 — Select Best Candidate
def detect_target(df):

    id_columns = detect_id_columns(df)

    scores = score_target_candidates(df, id_columns)

    if len(scores) == 0:
        return None, scores

    predicted_target = max(scores, key=scores.get)

    if scores[predicted_target] == 0:
        return None, scores

    return predicted_target, scores


In [6]:
# Step 4 — Problem Type Detection (Connected)
def detect_problem_type(df, target):

    if target is None:
        return "Unsupervised"

    unique_values = df[target].nunique()

    if df[target].dtype == "object" or unique_values < 50:
        return "Classification"

    return "Regression"


In [ ]:
def profile_dataset(df: pd.DataFrame, target: str) -> Dict[str, Any]:
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in dataset columns.")

    y = df[target]
    profile: Dict[str, Any] = {}

    profile["shape"] = {"rows": int(df.shape[0]), "cols": int(df.shape[1])}
    profile["columns"] = df.columns.astype(str).tolist()

    missing = (df.isna().mean() * 100).round(2).to_dict()
    profile["missing_pct"] = {str(k): float(v) for k, v in missing.items()}

    profile["target"] = str(target)
    profile["target_dtype"] = str(y.dtype)
    profile["problem_type"] = bool(detect_problem_type(df, target))

    # Feature types
    X = df.drop(columns=[target])
    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.astype(str).tolist()
    cat_cols = [c for c in X.columns.astype(str).tolist() if c not in numeric_cols]

    profile["feature_types"] = {"numeric": numeric_cols, "categorical": cat_cols}
    profile["n_unique_by_col"] = {str(c): int(df[c].nunique(dropna=True)) for c in df.columns.astype(str)}

    notes = []
    if profile["shape"]["rows"] < 1000:
        notes.append("Small dataset (<1000 rows): prefer simpler models / guard against overfitting.")
    if profile["shape"]["cols"] > 100:
        notes.append("High dimensionality (>100 columns): watch one-hot expansion and overfitting.")
    profile["notes"] = notes

    # Class balance if classification
    if profile["problem_type"] == "Classification":
        vc = y.value_counts(dropna=False)
        profile["class_counts"] = {str(k): int(v) for k, v in vc.items()}
        if len(vc) >= 2:
            ratio = float(vc.max() / max(vc.min(), 1))
        else:
            ratio = 1.0
        profile["imbalance_ratio"] = round(ratio, 3)
        if ratio >= 3.0:
            profile["notes"].append("Imbalance detected (ratio >= 3.0): prioritise macro metrics / balanced accuracy.")
    else:
        profile["class_counts"] = None
        profile["imbalance_ratio"] = None
        profile["notes"].append("Non-classification target detected: this template focuses on classification.")

    return profile